In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 06 \u2014 GMM Classification"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Objectives\n",
        "- Use GMM posteriors as probabilistic class predictions.\n",
        "- Compare boundaries with KMeans and Gaussian Naive Bayes.\n",
        "- Report accuracy and visual differences."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Mathematical Background\n",
        "For class-conditional Gaussian mixtures, classification selects the class with largest posterior probability. In this compact experiment, component labels are aligned to ground-truth classes for visualization."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Implementation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import sys\n",
        "from pathlib import Path\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "PROJECT = Path.cwd()\n",
        "if PROJECT.name == 'notebooks':\n",
        "    PROJECT = PROJECT.parent\n",
        "FIGURES = PROJECT / 'figures'\n",
        "FIGURES.mkdir(exist_ok=True)\n",
        "if str(PROJECT) not in sys.path:\n",
        "    sys.path.insert(0, str(PROJECT))\n",
        "\n",
        "plt.style.use('seaborn-v0_8-whitegrid')\n",
        "COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']\n",
        "RNG = np.random.default_rng(7)\n",
        "\n",
        "def savefig(name):\n",
        "    plt.tight_layout()\n",
        "    plt.savefig(FIGURES / name, dpi=180, bbox_inches='tight', transparent=True)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Experiments"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from sklearn.cluster import KMeans\n",
        "from sklearn.naive_bayes import GaussianNB\n",
        "from sklearn.metrics import accuracy_score\n",
        "from src.gaussian import sample_mixture\n",
        "from src.gmm import GaussianMixtureModel\n",
        "X,y=sample_mixture(np.array([.5,.5]), np.array([[-1.4,0],[1.4,.35]]), np.array([[[1.0,.75],[.75,1.0]],[[1.0,-.65],[-.65,1.0]]]), 1000, random_state=RNG)\n",
        "gmm=GaussianMixtureModel(2, random_state=2, max_iter=80).fit(X); pred=gmm.predict(X)\n",
        "if accuracy_score(y,pred)<.5: pred=1-pred\n",
        "km=KMeans(n_clusters=2, n_init=10, random_state=2).fit(X); kpred=km.labels_\n",
        "if accuracy_score(y,kpred)<.5: kpred=1-kpred\n",
        "gnb=GaussianNB().fit(X,y)\n",
        "print({'GMM': accuracy_score(y,pred), 'KMeans': accuracy_score(y,kpred), 'GaussianNB': accuracy_score(y,gnb.predict(X))})\n",
        "xx,yy=np.meshgrid(np.linspace(X[:,0].min()-1,X[:,0].max()+1,250),np.linspace(X[:,1].min()-1,X[:,1].max()+1,250))\n",
        "grid=np.column_stack([xx.ravel(),yy.ravel()])\n",
        "Z=gmm.predict(grid).reshape(xx.shape)\n",
        "plt.figure(figsize=(6,5)); plt.contourf(xx,yy,Z,alpha=.22,cmap='viridis'); plt.scatter(X[:,0],X[:,1],c=y,cmap='viridis',s=12,alpha=.65)\n",
        "plt.title('How does a GMM decision boundary bend with covariance?'); savefig('decision_boundary.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "fig, axes = plt.subplots(1,3,figsize=(13,4))\n",
        "models=[('GMM', pred),('KMeans', kpred),('GaussianNB', gnb.predict(X))]\n",
        "for ax,(name,p) in zip(axes,models):\n",
        "    ax.scatter(X[:,0],X[:,1],c=p,cmap='viridis',s=10,alpha=.6)\n",
        "    ax.set_title(f'{name}: accuracy={accuracy_score(y,p):.3f}')\n",
        "savefig('comparison.png')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Observations\n",
        "- GMM boundaries adapt to covariance; KMeans is driven by distance to centroids.\n",
        "- GaussianNB is simple and supervised but assumes feature independence within each class."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Key Takeaways\n",
        "- GMMs can be interpreted as density estimators, clusterers, and probabilistic classifiers.\n",
        "- Posterior probabilities provide both decisions and uncertainty."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Transition to the next notebook\n",
        "The project can now be extended to model selection, higher-dimensional data, and Bayesian mixtures."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "pygments_lexer": "ipython3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}